# Deduplication of Multiple Listings

Because the `Major` and `Primary` filters were deliberately not applied when
constructing the universe — they would have removed a substantial number of genuinely
delisted firms — the universe may contain more than one line per company.

Not all of these are duplicates. Two distinct situations must be separated:

| Situation | Periods | Action |
|---|---|---|
| **Concurrent listings** — the same firm quoted simultaneously on two venues, or two share lines | overlapping | keep one |
| **Re-listings** — a firm delisted and subsequently re-listed, or successive historical entities | disjoint | keep both |

Collapsing the second case would destroy exactly the delisted history the
survivorship-free construction is designed to preserve.

## Method

Candidate groups are securities sharing the same company name within the same
country. Within each group, listing periods are taken from the **data** — first and
last genuine observation, as established during padding removal — rather than from
reference metadata, which records only the start date.

Symbols are then clustered on two criteria jointly: two lines belong to the same
cluster only if their quotation periods **overlap** *and* they share the same **ICB
sector**. Each cluster represents one listing of one entity; **one line is retained
per cluster**, and clusters are retained in full.

The sector condition is what separates distinct companies sharing a name from
duplicate listings of the same company. Assystem and Brime Technologies merged in
2003 and appear under one name in different sectors; the Bolloré group had several
quoted entities; Kontron appears both as a hardware and as a software company. In
each case the lines are separate businesses with substantial independent histories,
and collapsing them would discard real data. Conversely, lines sharing both period
and sector — successive technical lines, dual venue listings — are genuine
duplicates.

Within a cluster the line retained is the one with the longest usable history,
with average market capitalisation as a tie-break.

## Note

This step is judgment-laden. The notebook is therefore organised so that the proposed
decisions are **inspected before being applied**, and the full list of discarded
securities is saved for the record.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import numpy as np
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Thesis/data"

monthly = pd.read_parquet(os.path.join(DATA_DIR, "clean_monthly.parquet"))
weekly  = pd.read_parquet(os.path.join(DATA_DIR, "clean_weekly.parquet"))
trunc   = pd.read_parquet(os.path.join(DATA_DIR, "truncation_points.parquet"))
master  = pd.read_csv(os.path.join(DATA_DIR, "master_universe.csv"), dtype=str)

trunc = trunc.set_index("symbol") if "symbol" in trunc.columns else trunc.set_index(trunc.columns[0])
print(f"monthly {len(monthly):>12,} rows | {monthly['symbol'].nunique():>6,} securities")
print(f"weekly  {len(weekly):>12,} rows | {weekly['symbol'].nunique():>6,} securities")
print(f"universe{len(master):>12,} rows")

monthly    6,732,454 rows |  6,185 securities
weekly     3,170,026 rows |  5,689 securities
universe       7,529 rows


## 2. Candidate groups

Securities sharing a company name within a country, restricted to those that
survived padding removal.

In [4]:
alive = set(monthly["symbol"])
u = master[master["Symbol"].isin(alive)].copy()
print(f"Securities with usable monthly data: {len(u):,}")

grp_key = ["country", "full_name"]
sizes = u.groupby(grp_key)["Symbol"].size()
dup_keys = sizes[sizes > 1]

cand = u.merge(dup_keys.rename("n_lines"), left_on=grp_key, right_index=True)
print(f"\nGroups with more than one line: {len(dup_keys):,}")
print(f"Securities involved            : {len(cand):,} "
      f"({100*len(cand)/len(u):.1f}% of the sample)")
print("\nGroup size distribution:")
print(dup_keys.value_counts().sort_index().rename("groups").to_frame().T.to_string())

Securities with usable monthly data: 6,185

Groups with more than one line: 93
Securities involved            : 285 (4.6% of the sample)

Group size distribution:
Symbol  2   3   4   6   7   8   9   10  11  13  16  20
groups  75   7   1   1   1   1   1   2   1   1   1   1


## 3. Listing periods

First and last genuine observation for each candidate, taken from the data.

In [5]:
cand = cand.join(trunc[["first_obs", "last_real"]], on="Symbol")

n_missing = int(cand["first_obs"].isna().sum())
print(f"Candidates without a period (no usable RI): {n_missing:,}")
if n_missing:
    print("  These are excluded from clustering and retained unchanged.")
cand = cand[cand["first_obs"].notna()].copy()

# average market capitalisation, used as tie-break
mv = (monthly[monthly["datatype"] == "MV"]
      .groupby("symbol")["value"].mean().rename("avg_mv"))
cand = cand.join(mv, on="Symbol")
cand["n_months"] = ((cand["last_real"] - cand["first_obs"]).dt.days / 30.44).round()

print(f"\nCandidates with a measurable period: {len(cand):,}")
print(cand[["n_months"]].describe().round(0).to_string())

Candidates without a period (no usable RI): 162
  These are excluded from clustering and retained unchanged.

Candidates with a measurable period: 123
       n_months
count     123.0
mean      101.0
std       107.0
min         1.0
25%        14.0
50%        60.0
75%       138.0
max       359.0


## 4. Clustering by period overlap

Two lines belong to the same episode if their quotation periods overlap by at least
`MIN_OVERLAP` of the shorter period. The threshold is deliberately low: a brief
overlap during a transition between share lines still indicates concurrency, whereas
a genuine re-listing shows no overlap at all.

In [7]:
MIN_OVERLAP = 0.25


def overlap_clusters(g, min_overlap=MIN_OVERLAP):
    """
    Cluster the lines of a name group into listing episodes.

    Two lines are merged only if their quotation periods overlap by at least
    `min_overlap` of the shorter period AND they report the same ICB sector.
    A differing sector indicates distinct businesses sharing a name rather than
    duplicate listings of one company; where the sector is unrecorded the lines
    are left unmerged, which errs towards retaining data.
    """
    g = g.reset_index(drop=True)
    n = len(g)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for i in range(n):
        for j in range(i + 1, n):
            si, sj = g.loc[i, "Sector"], g.loc[j, "Sector"]
            if pd.isna(si) or pd.isna(sj) or si != sj:
                continue                      # distinct entities, or unknown

            a1, b1 = g.loc[i, "first_obs"], g.loc[i, "last_real"]
            a2, b2 = g.loc[j, "first_obs"], g.loc[j, "last_real"]
            lo, hi = max(a1, a2), min(b1, b2)
            ov = max(0, (hi - lo).days)
            lmin = min((b1 - a1).days, (b2 - a2).days)
            share = ov / lmin if lmin > 0 else (1.0 if ov > 0 else 0.0)
            if share >= min_overlap:
                union(i, j)

    g["cluster"] = [find(i) for i in range(n)]
    return g


parts = []
for _, g in cand.groupby(grp_key):
    parts.append(overlap_clusters(g))
cl = pd.concat(parts, ignore_index=True)
cl["group_id"] = cl["country"] + "|" + cl["full_name"] + "|" + cl["cluster"].astype(str)

n_groups = cl.groupby(grp_key).ngroups
n_clusters = cl["group_id"].nunique()
print(f"Name groups            : {n_groups:,}")
print(f"Listing episodes       : {n_clusters:,}")
print(f"Securities to discard  : {len(cl) - n_clusters:,}")

Name groups            : 62
Listing episodes       : 94
Securities to discard  : 29


### Group composition

How groups split between concurrent listings, re-listings, and mixed cases.

In [8]:
summary = (cl.groupby(grp_key)
           .agg(n_lines=("Symbol", "size"), n_episodes=("cluster", "nunique"),
                n_sectors=("Sector", "nunique")))
summary["type"] = np.where(summary["n_episodes"] == 1, "single episode",
                    np.where(summary["n_episodes"] == summary["n_lines"],
                             "all distinct", "mixed"))
print(summary["type"].value_counts().to_string())

summary["removed"] = summary["n_lines"] - summary["n_episodes"]
print("\nLines removed by group type:")
print(summary.groupby("type")["removed"].sum().to_string())

multi = summary[summary["n_sectors"] > 1]
print(f"\nName groups spanning more than one sector: {len(multi):,}")
print("These are kept as separate entities rather than merged.")

type
all distinct      31
single episode    30
mixed              1

Lines removed by group type:
type
all distinct       0
mixed              2
single episode    27

Name groups spanning more than one sector: 10
These are kept as separate entities rather than merged.


## 5. Selection within each episode

The line with the longest usable history is retained; average market capitalisation
breaks ties.

In [9]:
cl = cl.sort_values(["group_id", "n_months", "avg_mv"],
                    ascending=[True, False, False])
cl["keep"] = ~cl.duplicated(subset="group_id", keep="first")

keep_syms = set(cl.loc[cl["keep"], "Symbol"])
drop_syms = set(cl.loc[~cl["keep"], "Symbol"])

print(f"Retained : {len(keep_syms):,}")
print(f"Discarded: {len(drop_syms):,}")

disc = cl.loc[~cl["keep"]]
print("\nDiscarded by country:")
print(disc["country"].value_counts().to_string())
print("\nDiscarded by status:")
print(disc["Activity"].value_counts().to_string())

Retained : 94
Discarded: 29

Discarded by country:
country
DE    13
FR    10
IT     5
ES     1

Discarded by status:
Activity
Dead      28
Active     1


### Inspection

Review before applying. Each block shows one name group with the retained line marked.

In [10]:
cols = ["Symbol", "Activity", "Sector", "first_obs", "last_real",
        "n_months", "avg_mv", "cluster", "keep"]
shown = 0
for (co, nm), g in cl.groupby(grp_key):
    if g["keep"].all():
        continue           # nothing discarded here
    print(f"\n{co} — {nm}")
    gg = g[cols].copy()
    gg["first_obs"] = gg["first_obs"].dt.strftime("%Y-%m")
    gg["last_real"] = gg["last_real"].dt.strftime("%Y-%m")
    gg["avg_mv"] = gg["avg_mv"].round(0)
    gg["Sector"] = gg["Sector"].str[:28]
    print(gg.to_string(index=False))
    shown += 1
    if shown >= 15:
        print(f"\n... ({(~cl['keep']).groupby([cl['country'], cl['full_name']]).any().sum() - shown} more groups)")
        break


DE — Adler Real Estate
Symbol Activity                       Sector first_obs last_real  n_months  avg_mv  cluster  keep
 D:ADL     Dead Real Estate Investment and S   1996-01   2023-11     334.0   280.0        0  True
D:ADLN     Dead Real Estate Investment and S   2017-02   2017-06       4.0     NaN        0 False

DE — Aixtron
Symbol Activity                       Sector first_obs last_real  n_months  avg_mv  cluster  keep
D:AIXA   Active Technology Hardware and Equi   1997-12   2025-12     336.0  1371.0        0  True
8784J9     Dead Technology Hardware and Equi   2014-04   2014-06       2.0     NaN        0 False

DE — Brenntag
Symbol Activity    Sector first_obs last_real  n_months  avg_mv  cluster  keep
 D:BNR   Active Chemicals   2010-04   2025-12     188.0  7657.0        0  True
2922X2     Dead Chemicals   2017-02   2017-07       5.0  4938.0        0 False

DE — CS PCC
Symbol Activity                       Sector first_obs last_real  n_months  avg_mv  cluster  keep
D:CCW1     

### Sanity check on discarded lines

A discarded line should not cover months that the retained line does not. Where it
does, the group deserves a closer look.

In [11]:
chk = []
for gid, g in cl.groupby("group_id"):
    if len(g) < 2:
        continue
    k = g[g["keep"]].iloc[0]
    for _, d in g[~g["keep"]].iterrows():
        before = max(0, (k["first_obs"] - d["first_obs"]).days // 30)
        after = max(0, (d["last_real"] - k["last_real"]).days // 30)
        if before + after > 12:
            chk.append({"group": gid, "kept": k["Symbol"], "dropped": d["Symbol"],
                        "months_before": before, "months_after": after})

chk = pd.DataFrame(chk)
print(f"Discarded lines extending more than 12 months beyond the retained one: {len(chk):,}")
if len(chk):
    print(chk.nlargest(10, ["months_before", "months_after"]).to_string(index=False))
    print("\nInspect these before applying; a large gap may indicate a re-listing "
          "that the overlap threshold has merged.")

Discarded lines extending more than 12 months beyond the retained one: 0


## 6. Apply

Run once the selection above has been reviewed.

In [12]:
monthly_d = monthly[~monthly["symbol"].isin(drop_syms)].copy()
weekly_d  = weekly[~weekly["symbol"].isin(drop_syms)].copy()

print("=== MONTHLY ===")
print(f"  rows      : {len(monthly):>12,} -> {len(monthly_d):>12,}")
print(f"  securities: {monthly['symbol'].nunique():>12,} -> {monthly_d['symbol'].nunique():>12,}")
print("=== WEEKLY ===")
print(f"  rows      : {len(weekly):>12,} -> {len(weekly_d):>12,}")
print(f"  securities: {weekly['symbol'].nunique():>12,} -> {weekly_d['symbol'].nunique():>12,}")

=== MONTHLY ===
  rows      :    6,732,454 ->    6,724,638
  securities:        6,185 ->        6,156
=== WEEKLY ===
  rows      :    3,170,026 ->    3,166,425
  securities:        5,689 ->        5,660


## 7. Save

In [13]:
monthly_d.to_parquet(os.path.join(DATA_DIR, "dedup_monthly.parquet"), index=False)
weekly_d.to_parquet(os.path.join(DATA_DIR, "dedup_weekly.parquet"), index=False)

log_cols = ["country", "full_name", "Symbol", "Activity", "Sector",
            "first_obs", "last_real", "n_months", "avg_mv", "cluster", "keep"]
cl[log_cols].to_csv(os.path.join(DATA_DIR, "deduplication_log.csv"), index=False)

for f in ["dedup_monthly.parquet", "dedup_weekly.parquet"]:
    p = os.path.join(DATA_DIR, f)
    print(f"  {f:28s} {len(pd.read_parquet(p)):>12,} rows  ({os.path.getsize(p)/1e6:.1f} MB)")
print(f"  {'deduplication_log.csv':28s} {len(cl):>12,} rows")
print("\nAnnual data are filtered at panel construction using the retained symbols.")

  dedup_monthly.parquet           6,724,638 rows  (15.1 MB)
  dedup_weekly.parquet            3,166,425 rows  (14.1 MB)
  deduplication_log.csv                 123 rows

Annual data are filtered at panel construction using the retained symbols.
